# Case by case analysis

The goal of this analysis is to expand on the previous extended analysis to perform a series of experiments to test the followig:
 - How does the LLM really affect the pipeline?
 - Is there any error in the audio pipeline? What happens if we remove it?
 - What happens if we expand scene graphs representations to more objects and scene information

In [ ]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
import json, os
import numpy as np
from matplotlib.patches import Patch
from classes.Plan import Plan

# Reflect
from main.data import load_data
from main.audio import process_sound
from main.execute_replan import run_correction_mem
from main.scene_graph import SceneGraph, Node as SceneGraphNode
from main.get_local_sg import get_scene_graph
from main.utils import get_label_from_object_id, convert_step_to_timestep, get_robot_plan_mem, get_scene_text_util, get_replan_prefix, get_admissible_actions
from main.exp import get_scene_text
from LLM.prompt import OpenAILLMPrompter
import main.constants as cons
from dataclasses import dataclass

In [ ]:
# Evaluation helpers 
def ts_to_sec(ts):
    """Convert 'MM:SS' timestamp string to total seconds (int)."""
    m, s = ts.strip().split(":")
    return int(m) * 60 + int(s)

def assess_error_localization(pred_list, gt_list) -> bool:
    if not pred_list or not gt_list:
        return False
    try:
        p_sec = ts_to_sec(pred_list[0])
        gt_secs = [ts_to_sec(t) for t in gt_list]
        if len(gt_secs) == 1:
            return p_sec == gt_secs[0]
        else:
            return gt_secs[0] <= p_sec <= gt_secs[-1]
    except Exception:
        return False

#  Data loading
def load_episode_data(sim_data_dir, episode_name):
    """Load all data for one episode; return (events, task_detail, object_list, interact_actions, nav_actions, data_path)."""
    task_name = episode_name.split('-')[0]
    data_path = os.path.join(sim_data_dir, task_name, episode_name)
    with open(os.path.join(data_path, "task.json"), 'r') as f:
        task_detail = json.load(f)
    events, task_detail, object_list, interact_actions, nav_actions = load_data(data_path, task_detail)
    return events, task_detail, object_list, interact_actions, nav_actions, data_path

def detect_episode_sounds(data_path, object_list):
    """Return detected sounds dict with 1-based step keys."""
    return {k + 1: v for k, v in process_sound(data_path, object_list).items()}

# Scene graph construction
def build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds):
    """Build global and per-step local scene graphs for an episode.

    Returns (global_sg, key_frames, local_graphs, object_list_set).
    """
    global_sg = SceneGraph(events[-1], task_detail)
    key_frames = set()
    local_graphs = {}
    prev_graph = SceneGraph(event=None, task=task_detail)
    total_points_dict, bbox3d_dict = {}, {}
    obj_held_prev = None
    count, interval = 0, 2
    action_frames = set(interact_actions) | set({idx[1] for idx in nav_actions.keys()})
    keyframe_triggers = action_frames | set(detected_sounds)

    for step_id, event in enumerate(events, 1):
        if step_id not in action_frames:
            count += 1
            if step_id not in detected_sounds and count % interval == 0:
                continue
        local_sg, total_points_dict, obj_held_prev, bbox3d_dict = get_scene_graph(
            step_id - 1, event, object_list, total_points_dict, bbox3d_dict, obj_held_prev, task_detail
        )
        local_graphs[step_id] = local_sg
        if local_sg != prev_graph:
            key_frames.add(step_id)
            prev_graph = local_sg
        if step_id in keyframe_triggers:
            key_frames.add(step_id)

    label_names = {label: get_label_from_object_id(label, events, task_detail) for label in total_points_dict}
    object_list_set = set(object_list)

    for label, name in label_names.items():
        if name is not None:
            global_sg.add_node_wo_edge(SceneGraphNode(
                name=name, object_id=label,
                pos3d=bbox3d_dict[label].get_center(),
                corner_pts=np.array(bbox3d_dict[label].get_box_points()),
                pcd=total_points_dict[label], global_node=True
            ))

    node_by_name = {node.name: node for node in global_sg.total_nodes}
    for label, name in label_names.items():
        if label.split("|")[0] in object_list_set and name in node_by_name:
            global_sg.add_node(node_by_name[name])

    global_sg.add_agent()
    return global_sg, key_frames, local_graphs, object_list_set

# Caption building
def _find_action_for_step(step_id_1based, interact_actions, nav_actions):
    """Helper to find action for a given step."""
    if step_id_1based in interact_actions:
        return interact_actions[step_id_1based], True
    for (min_step, max_step), action in nav_actions.items():
        if min_step <= step_id_1based <= max_step:
            return action, False
    return None, None

def build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds):
    """Build L1 (step-level) and L2 (subgoal-level) caption lists.

    Returns (L1_captions, L2_captions, state_summary_L1, state_summary_L2).
    L1_captions is a list of (step_id, caption_str) tuples.
    """
    L1_captions = []

    for step_id, _ in enumerate(events):
        if step_id + 1 not in local_graphs or step_id + 1 not in key_frames:
            continue
        action, _ = _find_action_for_step(step_id + 1, interact_actions, nav_actions)
        if not action:
            continue
        caption = f"{convert_step_to_timestep(step_id + 1, 1)}. Action: {action}. "
        caption += f"Visual observation: {get_scene_text(local_graphs[step_id + 1])}"
        # Use model-detected sounds (1-based keys) to match the validation pipeline
        if (step_id + 1) in detected_sounds:
            caption += f" Auditory observation: {detected_sounds[step_id + 1]}."
        caption += "\n"
        L1_captions.append((step_id + 1, caption))

    L2_captions = [cap.replace("Action", "Goal") for step_id, cap in L1_captions if step_id in interact_actions]
    state_summary_L1 = "".join(cap for _, cap in L1_captions)
    state_summary_L2 = "".join(L2_captions)
    return L1_captions, L2_captions, state_summary_L1, state_summary_L2

# LLM prompt utility
def build_prompt(info, **replacements):
    """Construct a {system, user} prompt dict from a template, applying keyword substitutions."""
    user_text = info['template-user']
    for key, value in replacements.items():
        user_text = user_text.replace(f"[{key.upper()}]", value)
    return {'system': info['template-system'], 'user': user_text}

# Reasoning pipeline
def run_reasoning(L1_captions, L2_captions, task, global_sg, llm_prompter, prompt_template):
    """Run subgoal verification → failure reasoning → GT labelling.

    Returns reasoning_dict.
    """
    L1_captions_text = [cap for _, cap in L1_captions]
    sv_info = prompt_template['subgoal-verifier']
    failed_caption = None

    for caption in L2_captions:
        subgoal = caption.split(". ")[1].split(": ")[1].lower()
        observation = caption[caption.find("Visual observation"):]
        prompt = build_prompt(sv_info, subgoal=subgoal, observation=observation)
        ans, _ = llm_prompter.query(prompt, sampling_params=sv_info['params'])
        if ans.split(", ")[0] != "Yes":
            failed_caption = caption
            break

    reasoning_dict = {}

    if failed_caption:  # execution error
        step_name = failed_caption.split(".")[0]
        for step_id, caption in L1_captions:
            if step_name not in caption:
                continue
            action = caption.split(". ")[1].split(": ")[1].lower()
            prev_observations = get_robot_plan_mem(L2_captions, L1_captions_text, step=step_name, with_obs=True)
            observation = caption[caption.find("Action"):]
            prompt_key = 'reasoning-execution' if prev_observations else 'reasoning-execution-no-history'
            re_info = prompt_template[prompt_key]
            re_prompt = build_prompt(re_info, action=action, task_name=task['name'],
                                     step=step_name, summary=prev_observations, observation=observation)
            failure_reason, _ = llm_prompter.query(prompt=re_prompt, sampling_params=re_info['params'])
            res_info = prompt_template['reasoning-execution-steps']
            res_prompt = build_prompt(res_info, failure_reason=failure_reason)
            time_steps, _ = llm_prompter.query(prompt=res_prompt, sampling_params=res_info['params'])
            reasoning_dict.update({
                'pred_failure_reason': failure_reason,
                'pred_failure_step':   [ts.strip(",") for ts in time_steps.split(", ")],
                'error_type':          'execution',
            })
            break

    else:  # planning error
        observation = get_robot_plan_mem(L2_captions, L1_captions_text, step=None, with_obs=False)
        rp_info = prompt_template['reasoning-plan']
        rp_prompt = build_prompt(rp_info, task_name=task['name'],
                                 success_condition=task['success_condition'],
                                 current_state=get_scene_text_util(global_sg),
                                 observation=observation)
        failure_reason, _ = llm_prompter.query(prompt=rp_prompt, sampling_params=rp_info['params'])
        rps_info = prompt_template['reasoning-plan-steps']
        rps_prompt = build_prompt(rps_info, prev_prompt=rp_prompt['user'] + " " + failure_reason)
        failure_step, _ = llm_prompter.query(prompt=rps_prompt, sampling_params=rps_info['params'])
        reasoning_dict.update({
            'pred_failure_reason': failure_reason,
            'pred_failure_step':   [failure_step.split(" ")[0].rstrip('.,')],
            'error_type':          'planning',
        })

    reasoning_dict.update({
        'gt_failure_reason': task['gt_failure_reason'],
        'gt_failure_step':   task['gt_failure_step'],
        'gt_error_type':     'planning' if task.get('chosen_failure') else 'execution',
    })
    return reasoning_dict

# Replanning
def get_initial_plan(actions):
    """Format the original plan, excluding navigation actions."""
    plan = ""
    idx = 0
    for action in actions:
        params = action[1:-1].split(", ")
        if params[0] != "navigate_to_obj":
            for i, obj_name in enumerate(params[1:]):
                params[i + 1] = cons.NAME_MAP.get(obj_name, obj_name.lower())
            plan += f"{idx + 1}. ({', '.join(params)})\n"
            idx += 1
    return plan[:-1]

def run_replanning(task, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template) -> Plan:
    """Generate a corrected plan given the failure reasoning.

    Returns replan_dict with keys: task_plan, llm_plan_raw, llm_plan, num_steps.
    """
    last_event = events[-1]
    current_state = get_scene_text_util(global_sg)
    global_object_list = list(
        set([obj["objectType"] for obj in last_event.metadata["objects"]]) | object_list_set
    )
    available_actions = get_admissible_actions(global_object_list, last_event)
    available_objects = sorted({token.strip("() ") for action in available_actions for token in action.split(", ")[1:]})
    corr_info = prompt_template['correction']
    prompt = {
        'system': corr_info['template-system'].replace(
            "[PREFIX]",
            get_replan_prefix(available_actions=available_actions, available_objects=available_objects),
        ),
        'user': (
            corr_info['template-user']
            .replace("[TASK_NAME]", task['name'])
            .replace("[PLAN]", get_initial_plan(task['actions']))
            .replace("[FAILURE_REASON]", reasoning_dict['pred_failure_reason'])
            .replace("[CURRENT_STATE]", current_state)
            .replace("[SUCCESS_CONDITION]", task['success_condition'])
        ),
    }
    print("Correction prompt:\n", prompt['system'] + "\n" + prompt['user'])
    plan, _ = llm_prompter.query(prompt=prompt, sampling_params=corr_info['params'], response_model=Plan)
    return {
        'task_plan': list(task['actions']),
        'llm_plan_raw': plan.model_dump()['actions'],
        'llm_plan': plan.actions,
        'num_steps': len(plan.actions),
    }

# Correction
def run_episode_correction(episode_name, task_name, data_path, task_detail, events, reasoning_dict, replan_dict):
    """Save reasoning and run the correction; returns correction_dict."""
    last_frame = len(events) - 1
    _reasoning_dir = os.path.join(data_path, 'state_summary', task_name, episode_name)
    os.makedirs(_reasoning_dir, exist_ok=True)
    with open(os.path.join(_reasoning_dir, 'reasoning.json'), 'w') as _rf:
        json.dump(reasoning_dict, _rf, default=str)
    return run_correction_mem(data_path, task_detail, events[-1], last_frame, replan_dict)

# Analysis output
def print_episode_analysis(episode_name, task, reasoning_dict, replan_dict):
    """Print a side-by-side comparison of GT vs. predicted reasoning and the correction plan."""
    print(f"Episode name: {episode_name}")
    print(f"Original task: {task['name']}")
    print(f"Original failure reason: {reasoning_dict['gt_failure_reason']}")
    print(f"Predicted failure reason: {reasoning_dict['pred_failure_reason']}")
    print(f"GT failure step(s): {reasoning_dict['gt_failure_step']}")
    print(f"Predicted failure step(s): {reasoning_dict['pred_failure_step']}")
    print(f"Side to side plans:\nOriginal plan:\n{replan_dict['llm_plan_raw']}")
    print("Correction plan:")
    for line, instruct in enumerate(replan_dict['llm_plan'], 1):
        print(f"{line}. {instruct}")

# Constants
SIM_DATA_DIR = "./datasets/sim_data/"


In [ ]:
# Get previous run data
from reflect.core.paths import sim_results_root

RESULTS_PATH = str(sim_results_root()) + "/"
PREVIOUS_RUN_PATH = os.path.join(RESULTS_PATH, "reflect_results_run3.json")
DATA_WITH_ERROR_LABELS = os.path.join(RESULTS_PATH, "error_taxonomy_all.csv")

with open(PREVIOUS_RUN_PATH, 'r') as f:
    previous_run_data = json.load(f)
previous_run_data = pd.DataFrame(previous_run_data)

# Drop empty columns
previous_run_data = previous_run_data.drop(columns=['global_sg'])

# Get human evaluation scores
EVAL_CSV_PATH = os.path.join(RESULTS_PATH, "exp_evaluation_run3.csv")
scored = pd.read_csv(EVAL_CSV_PATH)

# Join human scores with previous run data
previous_run_data = previous_run_data.merge(scored[['episode', 'human_score']], on='episode', how='left')

# Join error taxonomy codes
error_taxonomy_df = pd.read_csv(DATA_WITH_ERROR_LABELS)
previous_run_data['error'] = previous_run_data['error'].replace(r'^\s*$', pd.NA, regex=True)
previous_run_data['error'] = previous_run_data['error'].combine_first(
    previous_run_data['episode'].map(error_taxonomy_df.set_index('episode')['taxonomy_code'])
)

# Unfold dict columns for easier analysis
unfolded_reasoning = pd.json_normalize(previous_run_data['reasoning_dict'])
unfolded_replanning = pd.json_normalize(previous_run_data['replan_dict'])
unfolded_corrections = pd.json_normalize(previous_run_data['correction_dict'])

# Combine all data into a single DataFrame
run_df = pd.concat([previous_run_data.drop(columns=['reasoning_dict', 'replan_dict', 'correction_dict']),
                    unfolded_reasoning.add_prefix('reasoning_'),
                    unfolded_replanning.add_prefix('replan_'),
                    unfolded_corrections.add_prefix('correction_')], axis=1)

run_df = run_df[run_df['correction_success'] != True]


run_df.info()


# Test 1: How does the LLM really affect the pipeline?

In [ ]:
# Filter to only LLM-failed episodes for error analysis
error_episodes_df = run_df[(run_df['error'] == "RSN-DIAG") | (run_df['error'] == "RSN-PLAN")]

In [ ]:
# LLM setup Local
LLM_DIR     = "./LLM"
LOCAL_MODEL = "qwen3.5:9b"
OLLAMA_URL  = "http://localhost:11434"

#llm_prompter = LocalLLMPrompter(model_name=LOCAL_MODEL, base_url=OLLAMA_URL)
with open(os.path.join(LLM_DIR, "prompts.json")) as f:
    prompt_template = json.load(f)

# LLM setup OpenAI API
MODEL = "gpt-5.4"
API_KEY = os.environ["OPENAI_API_KEY"]  # set in your shell or .env (see .env.example)
llm_prompter = OpenAILLMPrompter(model=MODEL, api_key=API_KEY)

In [ ]:
@dataclass
class PromptLog:
    type: str
    system: str
    user: str

@dataclass
class Episode:
    task_type: str
    name: str
    l1_summary: str
    l2_summary: str
    num_keyframes: int
    num_events: int
    prompt_logs: list[PromptLog]
    error_label: str
    human_score: bool
    predicted_failure_reason: str
    predicted_failure_step: list[str]
    predicted_error_type: str
    ground_truth_failure_reason: str
    ground_truth_failure_step: list[str]
    ground_truth_error_type: str
    suggested_correction_plan_raw: list[str]
    suggested_correction_plan_translated: list[str]
    original_plan: list[str]


def _to_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, tuple):
        return list(value)
    if value is None:
        return []
    if pd.isna(value):
        return []
    return [value]


def _to_prompt_logs(prompt_logs_raw):
    if not isinstance(prompt_logs_raw, list):
        return []
    return [
        PromptLog(
            type=entry.get("call", ""),
            system=entry.get("prompt", {}).get("system", ""),
            user=entry.get("prompt", {}).get("user", ""),
        )
        for entry in prompt_logs_raw
    ]


episodes = [
    Episode(
        task_type=row["task"],
        name=row["episode"],
        l1_summary=row["l1_summary"],
        l2_summary=row["l2_summary"],
        num_keyframes=int(row["num_keyframes"]),
        num_events=int(row["num_events"]),
        prompt_logs=_to_prompt_logs(row["prompts_log"]),
        error_label=row["error"],
        human_score=bool(row["human_score"]),
        predicted_failure_reason=row["reasoning_pred_failure_reason"],
        predicted_failure_step=_to_list(row["reasoning_pred_failure_step"]),
        predicted_error_type=row["reasoning_error_type"],
        ground_truth_failure_reason=row["reasoning_gt_failure_reason"],
        ground_truth_failure_step=_to_list(row["reasoning_gt_failure_step"]),
        ground_truth_error_type=row["reasoning_gt_error_type"],
        suggested_correction_plan_raw=_to_list(row["correction_llm_plan_raw"]),
        suggested_correction_plan_translated=_to_list(row["correction_llm_plan"]),
        original_plan=_to_list(row["correction_task_plan"]),
    )
    for _, row in error_episodes_df.iterrows()
]

len(episodes)
episodes[0]

# Episode 1

In [ ]:
episode = episodes[0]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In this scenario, GPT was able to solve the problem.

Loc: True

Exp: True

Co-Plan: True

# Episode 2

In [ ]:
episode = episodes[1]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

The correction plan was correct but the translation failed 6. (open, microwave) => (toggle_on, Microwave) score: 0.8524898

Loc: True

Exp: True

Co-plan: False

# Episode 3

In [ ]:
episode = episodes[2]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

The LLM forgot to put down the plate in the correction plan even though it was explicitly stated in correction prompt.

Loc: True

Exp: True

Co-plan: False

# Episode 4

In [ ]:
episode = episodes[3]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

LLM Plan was correct but the translation layer failed: (open, microwave) => (toggle_on, Microwave) score: 0.8524898

Exp: True

Loc: True

Co-plan: False

# Episode 5

In [ ]:
episode = episodes[4]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

LLM Plan was sound but the translation layer failed: (open, microwave) => (toggle_on, Microwave) score: 0.8524898

Exp: True (With some hallucination)

Loc: True

Co-plan: Fail

# Episode 6

In [ ]:
episode = episodes[5]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

The LLM plan was correct but the translation layer failed: (open, microwave) => (toggle_on, Microwave) score: 0.8524898

Exp: True

Loc: True

Co-Plan: False

# Episode 7

In [ ]:
episode = episodes[6]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

LLM Plan was sound but the translation layer failed: (open, microwave) => (toggle_on, Microwave) score: 0.8524898

Exp: True

Loc: True

Co-plan: Fail

# Episode 8

In [ ]:
episode = episodes[7]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

The LLM did correctly assess that no error happened during execution, the success criteria wasn't specified correctly as the task goal. This is a False Positive

Exp: True

Loc: True

Co-plan: False

# Episode 9

In [ ]:
episode = episodes[8]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Task was successful.

Exp: True

Loc: True

Co-plan: True

# Episode 10

In [ ]:
episode = episodes[9]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

The task was impossible to solve for the LLM as:
 1. It did not defenitly know that a bowl cannot be placed in a cofee machine (Even though it guessed it in the explanation).
 2. The task goal did not specify what container should be used. It just said "a clean container contains coffe and it's in the countertop"
 3. The LLM was never informed about the affordances in the scene, so it didn't even know there was a mug in the scene.
 4. It wasn't able to localize that picking the bowl was wrong as it was the intended action.

Despite this, the plan itself did not make much sense, possibly because the input failure already specified there was no error present.

Exp: True

Loc: False (Technically)

Co-plan: False

# Episode 11

In [ ]:
episode = episodes[10]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Task was successful.

Exp: True

Loc: True

Co-Plan: True

# Episode 12

In [ ]:
episode = episodes[11]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Multiple things went wrong in this scenario:
 1. The perception layer did not identify that the apple slice was inside bowl at 02:20, maiking the LLM incorrectly stop the sequence as an execution error.
 2. The LLM failed to infer that the bowl that would be using was already filled in from the previous run. (The LLM made the incorrect assumtion that either the bowl would be empty or that there would be another bowl)

 Exp: True

 Loc: False

 Co-plan: False

# Episode 13

In [ ]:
episode = episodes[12]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Task success.

Exp: True

Loc: True

Co-plan: True

# Episode 14

In [ ]:
episode = episodes[13]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

The LLM did partially explained the failure, it localized what the error was, but not that i was a failure in the rebot grasping the bowl and it later hallucinated. Additionally, the proposed plan was also wrong.

Exp: True (Debatable)

Loc: True

Co-plan: False


# Episode 15

In [ ]:
episode = episodes[14]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Task success:

Exp: True

Loc: True

Co-plan: True

# Episode 16

In [ ]:
episode = episodes[15]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

The LLM failed to create tha plan, I'm starting to notice a pattern, the LLM tries to copy the original plan or, at least uses it as example of what would be correct, possibly because of this part in the re-planning prompt: 

The plan should 1) not contain any if statements 2) contain only the available actions **3) resemble the format of the initial plan.**

Exp: True

Loc: True

Co-plan: False

# Episode 17

In [ ]:
episode = episodes[16]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

This is a false negative, even though the task was successfully re-planned, the perception layer provided the wrong edges for egg, which made the LLM falg an error at that point and create a false error explanation. As the LLM re-planned with common-sense, the task still succeded. 

Exp: False

Loc: False

Co-Plan: True

# Episode 18

In [ ]:
episode = episodes[17]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Exactly same case and frame as the previous episode, the perception layer miss-identified the edges in regards to the egg. Making the LLM flag a failure where there wasn't one and creating a false failure explanation. Sub-sequent plan succeded as the LLM applied common-sense, not because it identified and fixed the error. This is a false negative.

Exp: False

Loc: False

Co-plan: True

# Episode 19

In [ ]:
episode = episodes[18]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Task success.

Exp: True

Loc: True

Co-plan: True

# Episode 20

In [ ]:
episode = episodes[19]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Task success.

Exp: True

Loc: True

Co-plan: True

# Episode 21

In [ ]:
episode = episodes[20]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

This is a False Positive, the LLM correctly identified the problem and proposed a sound plan, but the action primitives/simulation failde to correctly perform the tasks.

Exp: True

Loc: True

Co-plan: False

# Episode 22

In [ ]:
episode = episodes[21]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
for L2 in L2_captions:
    print(L2)

LLM miss-flagged step at 00:42 as failure, even though the scene graph description was correct.

Exp: False

Loc: False

Co-plan: False

# Episode 23

In [ ]:
episode = episodes[22]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

LLM struggled to keep the current state of the scene into account when generating the correction plan, it forgot to take into account that bread was on top of toaster.

Exp: True

Loc: True

Co-plan: False

# Episode 24

In [ ]:
episode = episodes[23]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Same as previous episode, LLM forgot to take into account that in the current state, bread is sitting on top of toaster.

Exp: True

Loc: True

Co-plan: False

# Episode 25

In [ ]:
episode = episodes[24]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Same as previous episodes, the LLM forgot to take into consideration that in the current state, the bread was on top of the toaster so the plan did not take it into consideration.

Exp: True

Loc: True

Co-plan: False

# Episode 26

In [ ]:
episode = episodes[25]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

The LLM failed to infer that a non-sliced bread cannot be put inside the toaster.

Exp: False

Loc: False

Co-plan: True

# Episode 27

In [ ]:
episode = episodes[26]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Task success.

Exp: True

Loc: True

Co-plan: True

# Episode 28

In [ ]:
episode = episodes[27]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Task success.

Exp: True

Loc: True

Co-plan: True

# Episode 29

In [ ]:
episode = episodes[28]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Task success (Even though the explanation could have been better).

Exp: True

Loc: True

Co-plan: True

# Episode 30

In [ ]:
episode = episodes[29]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

The LLM reasoned consistently, but after digging into the auditory contradiction that made the LLM infer the wrong failure, I discovered that the problem is that when we inject the sound, it spills over frame 00:26 and later, since fram 26 is still over the auditory threshold, the classifier puts the same label to it. In summary:
inject waveform -> encode MP4/AAC -> decode again -> threshold 1-second bins

Encoding with AAC makes the ends of the injected waveform not "perfect" anymore as it works by reconstructing the waveform with coefficients that can spill from one block to another, essentially it makes it perceptually near-perfect but the sample is not the same anymore. So they spill into the neighbouring 1-second bins that the classifiers later labels with the exact same label.

Exp: True

Loc: True

Co-plan: True

# Episode 31

In [ ]:
episode = episodes[30]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Task success.

Exp: True

Loc: True

Co-plan: True

# Episode 32

In [ ]:
episode = episodes[31]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Same error as previous episode, audio fuzziness + not coarse-enough audio labeler

Exp: False

Loc: False

Co-plan: False

# Episode 33

In [ ]:
episode = episodes[32]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

Same as the previous audio problem, the pipeline creates fuziness that leaks into the 1 second bins.

Exp: False

Loc: False

Co-plan: False

# Episode 34

In [ ]:
episode = episodes[33]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

The LLM plan was correct but but the translation layer failed (pour, pot, house plant) => (pour, HousePlant, Pot) score: 0.9812078.

Exp: True

Loc: True

Co-plan: False

# Episode 35

In [ ]:
episode = episodes[34]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

The LLM did not localize the error correctly as it had the problem with the audio leaking into neighbouring boxes, despite this it still managed to recover by applying a standard pipeline again.

Exp: False

Loc: False

Co-plan: True

## Test 1: Findings & Conclusions

In [ ]:
loc_rate = 26 / len(episodes) * 100
exp_rate = 28 / len(episodes) * 100
co_plan_rate = 16 / len(episodes) * 100

# Function to get localization
def get_localization_success(gt, pred):
    if isinstance(gt, str):
        gt = [gt]
    
    if isinstance(pred, str):
        pred = [pred]
    
    for pred_item in pred:
        if pred_item in gt:
            return True
    return False

loc_rate_previous = error_episodes_df.apply(lambda row: get_localization_success(row['reasoning_gt_failure_step'], row['reasoning_pred_failure_step']), axis=1).mean() * 100
exp_rate_previous = (error_episodes_df['human_score'] == 1.0).mean() * 100
co_plan_rate_previous = (error_episodes_df['correction_success'] == True).mean() * 100

colors = plt.get_cmap('Set1').colors

# Plot current vs previous rates
metrics = ['Localization', 'Explanation', 'Correction Plan']
current_rates = [loc_rate, exp_rate, co_plan_rate]
previous_rates = [loc_rate_previous, exp_rate_previous, co_plan_rate_previous]
x = np.arange(len(metrics))
width = 0.35
fig, ax = plt.subplots( figsize=(20, 12) )
rects1 = ax.bar(x - width/2, current_rates, width, label='Current', color=colors)
rects2 = ax.bar(x + width/2, previous_rates, width, label='Previous', color=colors, alpha=0.5, hatch='//')
for rect in rects1 + rects2:
    height = rect.get_height()
    ax.annotate(f'{height:.1f}%',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom')
ax.set_ylabel('Success Rate (%)')
ax.set_title('GPT-5.4 VS QWEN3.5:9B Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
legend_handles = [
    Patch(facecolor='lightgray', edgecolor='black', label='Current'),
    Patch(facecolor='lightgray', edgecolor='black', hatch='//', label='Previous')
]

ax.legend(handles=legend_handles)
plt.ylim(0, 100)
plt.show()

### Error Type Summary

| Error type | Count | Interpretation |
| --- | ---: | --- |
| Task success / no meaningful error | 11 | Episode was broadly successful, with no substantive failure mode affecting correction performance. |
| Translation layer failure | 6 | The reasoning or plan was mostly correct, but action translation mapped it incorrectly. |
| LLM state tracking | 5 | The LLM failed to maintain the current scene state into account while generating the correction plan. Leading to wrong or incomplete plans. |
| Audio label leakage / audio fuzziness | 4 | Audio preprocessing or labeling leaked into neighboring time bins and introduced misleading evidence. |
| Perception scene graph error | 3 | Incorrect or missing scene relations from perception layer. |
| Task specification or affordance gap | 2 | The task goal, available affordances, or scene constraints were underspecified for reliable correction. |
| LLM false failure localization / hallucinated explanation | 2 | The LLM diagnosed the wrong failure point or generated an unsupported explanation. |
| Evaluation success criteria mismatch | 1 | The evaluation framing did not cleanly match the real task objective. |
| Execution primitive or simulator failure | 1 | A reasonable correction could not be executed reliably because of low-level simulator or primitive failure. |

### Conclusions

- Failures were distributed across the full pipeline rather than concentrated in a single module.
- Translation and action grounding were one of the clearest engineering bottlenecks, since several episodes had correct reasoning but incorrect action execution.
- Replanning under changing scene state remains difficult for the LLM, especially when it must account for already-completed actions or current object placement.
- Perception errors strongly affected explanation quality, showing that some apparent reasoning failures were actually caused by incorrect upstream inputs.
- Audio-label leakage emerged as a repeated and concrete failure mode, suggesting that improving audio preprocessing could remove several downstream errors.
- Some episodes reflected benchmark or task-design limitations, including underspecified goals, missing affordance information, or mismatched success criteria.
- The system could sometimes recover even when the explanation was wrong, which suggests that recovery ability and explanation accuracy are related but not identical.
- The number of clean success cases shows that the overall approach is viable when multimodal inputs and action grounding are reliable.

In [ ]:
error_summary = pd.DataFrame({
    'Error type': [
        'Translation layer failure',
        'LLM state tracking / replanning failure',
        'Audio label leakage / audio fuzziness',
        'Perception scene graph error',
        'Task specification or affordance gap',
        'LLM false failure localization / hallucinated explanation',
        'Evaluation success criteria mismatch',
        'Execution primitive or simulator failure'
    ],
    'Count': [6, 5, 4, 3, 2, 2, 1, 1]
})

error_summary['Percentage'] = error_summary['Count'] / error_summary['Count'].sum() * 100
plot_df = error_summary.sort_values('Count', ascending=True)

colors = ['#C75C5C' for label in plot_df['Error type']]

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(plot_df['Error type'], plot_df['Count'], color=colors)

for bar, count, pct in zip(bars, plot_df['Count'], plot_df['Percentage']):
    ax.text(bar.get_width() + 0.12, bar.get_y() + bar.get_height() / 2,
            f'{count} ({pct:.1f}%)', va='center', fontsize=11)

ax.set_title('Distribution of Error Types', fontsize=16)
ax.set_xlabel('Number of episodes')
ax.set_ylabel('')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', linestyle='--', alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


### Example Episodes Per Error Type

**Translation layer failure**

- heatPotato-1: The correction plan was considered correct, but the translation layer incorrectly mapped (open, microwave) to (toggle_on, Microwave).
- waterPlant-4: The correction plan was considered correct, but the translation layer incorrectly mapped (pour, pot, housePlant) to (pour, housePlant, pot)

**LLM state tracking or replanning failure**

- heatPotato-2: The LLM forgot to include putting down the plate in the correction plan.
- toastBread-2: The LLM failed to keep track of the current scene state and forgot that the bread was already on top of the toaster.

**Audio label leakage / audio fuzziness**

- warmWater-7: Audio injected into the episode leaked into neighboring one-second bins after encoding and decoding, which caused repeated incorrect labels and a misleading failure explanation.
- waterPlant-10: The same audio-pipeline issue appeared again, where fuzziness leaked into adjacent time bins and created a false contradiction for the LLM.

**Perception scene graph error**

- makeSalad-3: The perception layer missed that the apple slice was already inside the bowl, which caused the LLM to incorrectly stop the sequence and explain a failure that was not actually there.
- storeEgg-4: The perception layer provided incorrect edge information for the egg, which caused the LLM to flag an error and produce a false explanation even though replanning still succeeded.

**Task specification or affordance gap**

- makeCoffee-9: The task goal only required a clean container with coffee, but it did not specify which container should be used, and the LLM was not given enough affordance information to know that a mug was available and a bowl was inappropriate.
- toastBread-7: The LLM did not infer that a non-sliced bread could not be inserted into the toaster, showing a gap in available affordance knowledge.

**LLM false failure localization / hallucinated explanation**

- makeSalad-6: The LLM partially localized the issue, but it missed that the real problem was a robot grasp failure and then hallucinated part of the explanation.
- switchDevices-6: The LLM flagged step 00:42 as a failure even though the scene graph description was correct, so the diagnosis itself was wrong.

**Evaluation success criteria mismatch**

- heatPotato-7: The LLM correctly reasoned that no execution error occurred, but the episode was still treated as problematic because the success criteria did not align well with the real task goal.

**Execution primitive or simulator failure**

- switchDevices-10: The LLM correctly identified the problem and proposed a sound correction plan, but the action primitives or simulator failed to carry it out successfully.


## Test after fixing audio and plan translation

Failed episodes because of plan translation in round 1: 
 * Episode 2: heatPotato-1
 * Episode 4: heatPotato-5
 * Episode 5: heatPotato-4
 * Episode 6: heatPotato-3
 * Episode 7: heatPotato-6
 * Episode 34: waterPlant-4

Audio-related failures:

 * Episode 32: warmWater-8
 * Episode 33: waterPlant-10
 * Episode 35: warmWater-9

In [ ]:
failed_episodes_audio = [episodes[i] for i in [31, 32, 34]]
failed_episodes_translation = [episodes[i] for i in [1, 3, 4, 5, 6, 33]]
episodes_to_retry = failed_episodes_audio + failed_episodes_translation

# Preserve order while avoiding duplicate episodes.
seen_episode_names = set()
episodes_to_retry = [
    episode for episode in episodes_to_retry
    if not (episode.name in seen_episode_names or seen_episode_names.add(episode.name))
]

retry_results = []

for episode in episodes_to_retry:
    print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
    try:
        # Get episode data
        events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
        print("Loaded episode data.")

        # Get detected sounds
        detected_sounds = detect_episode_sounds(data_path, object_list)
        print(f"Detected sounds: {detected_sounds}")

        # Build scene graphs
        global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
        print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

        # Build captions
        L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
        print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

        # Run reasoning
        reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
        print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

        # Run replanning
        replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
        print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

        # Run correction
        correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
        print(f"Completed correction. Correction success: {correction_dict.get('success', False)}")

        print_episode_analysis(episode.name, task_detail, reasoning_dict, correction_dict)

        retry_results.append({
            'episode': episode.name,
            'task_type': episode.task_type,
            'error_label': episode.error_label,
            'human_score': episode.human_score,
            'retry_group': 'audio' if episode in failed_episodes_audio else 'translation',
            'pred_failure_reason': reasoning_dict.get('pred_failure_reason', ''),
            'pred_failure_step': reasoning_dict.get('pred_failure_step', []),
            'gt_failure_reason': reasoning_dict.get('gt_failure_reason', ''),
            'gt_failure_step': reasoning_dict.get('gt_failure_step', []),
            'llm_plan_raw': correction_dict.get('llm_plan_raw', []),
            'llm_plan': [str(step) for step in correction_dict.get('llm_plan', [])],
            'task_plan': correction_dict.get('task_plan', []),
            'num_steps': correction_dict.get('num_steps', 0),
            'correction_success': correction_dict.get('success', False),
        })
    except Exception as exc:
        print(f"Episode failed during retry: {type(exc).__name__}: {exc}")
        retry_results.append({
            'episode': episode.name,
            'task_type': episode.task_type,
            'error_label': episode.error_label,
            'human_score': episode.human_score,
            'retry_group': 'audio' if episode in failed_episodes_audio else 'translation',
            'pred_failure_reason': None,
            'pred_failure_step': [],
            'gt_failure_reason': episode.ground_truth_failure_reason,
            'gt_failure_step': episode.ground_truth_failure_step,
            'llm_plan_raw': [],
            'llm_plan': [],
            'task_plan': episode.original_plan,
            'num_steps': 0,
            'correction_success': False,
            'exception_type': type(exc).__name__,
            'exception_message': str(exc),
        })
    print('-' * 120)

retry_results_df = pd.DataFrame(retry_results)
retry_results_df


In [ ]:
def format_plan(plan):
    if not plan:
        return "No plan generated."
    formatted_steps = []
    for step in plan:
        if isinstance(step, str):
            formatted_steps.append(step)
        else:
            formatted_steps.append(str(step))
    return "\n".join(formatted_steps)

# Episode analysis for retry episodes one by one
episode = retry_results_df.iloc[8]
print(f"Analyzing episode: {episode['episode']} with error label: {episode['error_label']} and human score: {episode['human_score']}")
print(f"Predicted failure reason: {episode['pred_failure_reason']}, \nPredicted failure step: {episode['pred_failure_step']}")
print(f"Ground truth failure reason: {episode['gt_failure_reason']}, \nGround truth failure step: {episode['gt_failure_step']}")
print(f"\nLLM suggested plan (parsed): \n{format_plan(episode['llm_plan'])}")
print(f"\nTask plan: \n{format_plan(episode['task_plan'])}")
print(f"Correction success: {episode['correction_success']}")


warmWater-8: The LLM decided to pour the filled in mug to the sink
 - Loc: False
 - Exp: True
 - Co-Plan: False

waterPlant-10: Correct
 - Loc: True
 - Exp: True
 - Co-plan: True

warmWater-9: The plan and executions where sound, the simulator/evaluator failed to capture the final state correctly, adding actions that wern't present in the plan at the end. False positive
 - Loc: True
 - Exp: True
 - Co-plan: False

heatPotato-1: The LLM didn't take into account current state of microwave
 - Loc: True
 - Exp: True
 - Co-plan: False

heatPotato-5: Correct
 - Loc: True
 - Exp: True
 - Co-plan: True

heatPotato-4: LLM didn't take into account that in order to open microwave it needs to turn off first. This is a simulator constrained, plan would have probably worked in reality.
 - Loc: True
 - Exp: True
 - Co-plan: False

heatPotato-3: Correct
 - Loc: True
 - Exp: True
 - Co-plan: True

heatPotato-6: Correct
 - Loc: True
 - Exp: True
 - Co-plan: True

waterPlant-4: Correct
 - Loc: True
 - Exp: True
 - Co-plan: True

In [ ]:
loc_rate_retry = 28 / len(episodes) * 100
exp_rate_retry = 31 / len(episodes) * 100
co_plan_rate_retry = 20 / len(episodes) * 100

# Plot current vs before fixing audio and translation issues
metrics = ['Localization', 'Explanation', 'Correction Plan']
current_rates = [loc_rate_retry, exp_rate_retry, co_plan_rate_retry]
previous_rates = [loc_rate, exp_rate, co_plan_rate]
x = np.arange(len(metrics))
width = 0.35
fig, ax = plt.subplots( figsize=(20, 12) )
rects1 = ax.bar(x - width/2, current_rates, width, label='After Fixing', color=colors)
rects2 = ax.bar(x + width/2, previous_rates, width, label='Before Fixing', color=colors, alpha=0.5, hatch='//')
for rect in rects1 + rects2:
    height = rect.get_height()
    ax.annotate(f'{height:.1f}%',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                fontsize=12)
ax.set_ylabel('Success Rate (%)')
ax.set_title('GPT-5.4 Performance Before vs After Fixing Audio and Translation Issues', fontsize=16)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
legend_handles = [
    Patch(facecolor='lightgray', edgecolor='black', label='After Fixing'),
    Patch(facecolor='lightgray', edgecolor='black', hatch='//', label='Before Fixing')
]
ax.legend(handles=legend_handles)
plt.ylim(0, 100)
plt.show()

In [ ]:
# Issue-type counts before vs after the audio / translation fixes
# Assumptions for the updated taxonomy:
# - The retried translation episodes are reclassified using the manual notes above.
# - `heatPotato-4` is counted as an LLM state-tracking / replanning failure.
# - `warmWater-9` is counted as an evaluation success-criteria mismatch.
# - `warmWater-7` is removed from the audio bucket because it reflects the same
#   leakage artifact that was fixed, even though that original run already succeeded.

issue_types = [
    'Translation layer failure',
    'LLM state tracking / replanning failure',
    'Audio label leakage / audio fuzziness',
    'Perception scene graph error',
    'Task specification or affordance gap',
    'LLM false failure localization / hallucinated explanation',
    'Evaluation success criteria mismatch',
    'Execution primitive or simulator failure',
]

issue_counts_before = {
    'Translation layer failure': 6,
    'LLM state tracking / replanning failure': 5,
    'Audio label leakage / audio fuzziness': 4,
    'Perception scene graph error': 3,
    'Task specification or affordance gap': 2,
    'LLM false failure localization / hallucinated explanation': 2,
    'Evaluation success criteria mismatch': 1,
    'Execution primitive or simulator failure': 1,
}

issue_counts_after = {
    'Translation layer failure': 0,
    'LLM state tracking / replanning failure': 8,
    'Audio label leakage / audio fuzziness': 0,
    'Perception scene graph error': 3,
    'Task specification or affordance gap': 2,
    'LLM false failure localization / hallucinated explanation': 2,
    'Evaluation success criteria mismatch': 2,
    'Execution primitive or simulator failure': 1,
}

issue_count_comparison = pd.DataFrame({
    'Error type': issue_types,
    'After Fixing': [issue_counts_after[issue] for issue in issue_types],
    'Before Fixing': [issue_counts_before[issue] for issue in issue_types],
})
issue_count_comparison['Delta'] = issue_count_comparison['After Fixing'] - issue_count_comparison['Before Fixing']
issue_count_comparison

plot_df = issue_count_comparison.sort_values('Before Fixing', ascending=True)
y = np.arange(len(plot_df))
height = 0.35

fig, ax = plt.subplots(figsize=(16, 8))
bars_after = ax.barh(y - height / 2, plot_df['After Fixing'], height, label='After Fixing', color='#C75C5C')
bars_before = ax.barh(y + height / 2, plot_df['Before Fixing'], height, label='Before Fixing', color='#C75C5C', alpha=0.45, hatch='//')

for bars in (bars_after, bars_before):
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.08, bar.get_y() + bar.get_height() / 2, f'{int(width)}', va='center', fontsize=11)

ax.set_yticks(y)
ax.set_yticklabels(plot_df['Error type'])
ax.set_xlabel('Number of episodes')
ax.set_ylabel('')
ax.set_title('Issue Type Counts Before vs After Fixing Audio and Translation Issues', fontsize=16)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', linestyle='--', alpha=0.3)
ax.set_axisbelow(True)
legend_handles = [
    Patch(facecolor='lightgray', edgecolor='black', label='After Fixing'),
    Patch(facecolor='lightgray', edgecolor='black', hatch='//', label='Before Fixing'),
]
ax.legend(handles=legend_handles)
ax.set_xlim(0, max(plot_df['Before Fixing'].max(), plot_df['After Fixing'].max()) + 1.5)
plt.tight_layout()
plt.show()


# Test 2: Can we fix the audio problem? How does it evolve when we fix it?

In [ ]:
# Filter to only LLM-failed episodes for error analysis
error_episodes_df = run_df[(run_df['error'] == "PER-AUDIO")]

In [ ]:
# LLM setup Local
LLM_DIR     = "./LLM"
LOCAL_MODEL = "qwen3.5:9b"
OLLAMA_URL  = "http://localhost:11434"

#llm_prompter = LocalLLMPrompter(model_name=LOCAL_MODEL, base_url=OLLAMA_URL)
with open(os.path.join(LLM_DIR, "prompts.json")) as f:
    prompt_template = json.load(f)

# LLM setup OpenAI API
MODEL = "gpt-5.4"
API_KEY = os.environ["OPENAI_API_KEY"]  # set in your shell or .env (see .env.example)
llm_prompter = OpenAILLMPrompter(model=MODEL, api_key=API_KEY)

In [ ]:
@dataclass
class PromptLog:
    type: str
    system: str
    user: str

@dataclass
class Episode:
    task_type: str
    name: str
    l1_summary: str
    l2_summary: str
    num_keyframes: int
    num_events: int
    prompt_logs: list[PromptLog]
    error_label: str
    human_score: bool
    predicted_failure_reason: str
    predicted_failure_step: list[str]
    predicted_error_type: str
    ground_truth_failure_reason: str
    ground_truth_failure_step: list[str]
    ground_truth_error_type: str
    suggested_correction_plan_raw: list[str]
    suggested_correction_plan_translated: list[str]
    original_plan: list[str]
    loc: bool = False
    exp: bool = False
    co_plan: bool = False
    pipeline_error: str = None


def _to_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, tuple):
        return list(value)
    if value is None:
        return []
    if pd.isna(value):
        return []
    return [value]


def _to_prompt_logs(prompt_logs_raw):
    if not isinstance(prompt_logs_raw, list):
        return []
    return [
        PromptLog(
            type=entry.get("call", ""),
            system=entry.get("prompt", {}).get("system", ""),
            user=entry.get("prompt", {}).get("user", ""),
        )
        for entry in prompt_logs_raw
    ]


episodes = [
    Episode(
        task_type=row["task"],
        name=row["episode"],
        l1_summary=row["l1_summary"],
        l2_summary=row["l2_summary"],
        num_keyframes=int(row["num_keyframes"]),
        num_events=int(row["num_events"]),
        prompt_logs=_to_prompt_logs(row["prompts_log"]),
        error_label=row["error"],
        human_score=bool(row["human_score"]),
        predicted_failure_reason=row["reasoning_pred_failure_reason"],
        predicted_failure_step=_to_list(row["reasoning_pred_failure_step"]),
        predicted_error_type=row["reasoning_error_type"],
        ground_truth_failure_reason=row["reasoning_gt_failure_reason"],
        ground_truth_failure_step=_to_list(row["reasoning_gt_failure_step"]),
        ground_truth_error_type=row["reasoning_gt_error_type"],
        suggested_correction_plan_raw=_to_list(row["correction_llm_plan_raw"]),
        suggested_correction_plan_translated=_to_list(row["correction_llm_plan"]),
        original_plan=_to_list(row["correction_task_plan"]),
    )
    for _, row in error_episodes_df.iterrows()
]

len(episodes)
episodes[0]

# Episode 1

In [ ]:
episode = episodes[0]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[0]
episode.co_plan = True
episode.exp = True
episode.loc = True

# Episode 2

In [ ]:
episode = episodes[1]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[1]
episode.loc = True
episode.exp = True
episode.co_plan = True

# Episode 3

In [ ]:
episode = episodes[2]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[2]
episode.loc = True
episode.exp = True
episode.co_plan = True

# Episode 4

In [ ]:
episode = episodes[3]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[3]
episode.loc = True
episode.exp = True
episode.co_plan = True

# Episode 5

In [ ]:
episode = episodes[4]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[4]
episode.loc = True
episode.exp = True
episode.co_plan = True

# Episode 6

In [ ]:
episode = episodes[5]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[5]
episode.loc = True
episode.exp = True
episode.co_plan = True

# Episode 7

In [ ]:
episode = episodes[6]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[6]
episode.loc = True
episode.exp = True
episode.co_plan = False
episode.pipeline_error = "Simulator failed to correctly execute the action primitive but plan was potentially correct"

# Episode 8

In [ ]:
episode = episodes[7]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[7]
episode.loc = False
episode.exp = False
episode.co_plan = False
episode.pipeline_error = "There where two mugs available in the scenario of which the LLM wasn't informed of, which made the later execution of a potentially correct plan fail as the action primitive selected the other mug."

# Episode 9

In [ ]:
episode = episodes[8]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[8]
episode.loc = True
episode.exp = True
episode.co_plan = True

# Episode 10

In [ ]:
episode = episodes[9]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[9]
episode.loc = True
episode.exp = True
episode.co_plan = True

# Episode 11

In [ ]:
episode = episodes[10]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[10]
episode.loc = True
episode.exp = False
episode.co_plan = False
episode.pipeline_error = "The perception layer failed to inform the LLM about the presence of a lettuce in front of the egg"

# Episode 12

In [ ]:
episode = episodes[11]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[11]
episode.loc = True
episode.exp = True
episode.co_plan = False
episode.pipeline_error = "The evaluation failed as the plan was correct and completed correctly"

# Episode 13

In [ ]:
episode = episodes[12]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[12]
episode.loc = True
episode.exp = False
episode.co_plan = False
episode.pipeline_error = "The perception layer failed to inform the LLM about the presence of a lettuce in front of the egg"

# Episode 14

In [ ]:
episode = episodes[13]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[13]
episode.loc = True
episode.exp = True
episode.co_plan = True

# Episode 15

In [ ]:
episode = episodes[14]

print(f"Processing episode: {episode.name} with error label: {episode.error_label} and human score: {episode.human_score}")
# Get episode data
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode.name)
print("Loaded episode data.")

# Get detected sounds
detected_sounds = detect_episode_sounds(data_path, object_list)
print(f"Detected sounds: {detected_sounds}")

# Build scene graphs
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds)
print(f"Built scene graphs. Global SG nodes: {len(global_sg.total_nodes)}, key frames: {len(key_frames)}")

# Build captions
L1_captions, L2_captions, _, _ = build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds)
L1_captions_text = [cap for _, cap in L1_captions]
print(f"Built captions. L1 captions: {len(L1_captions)}, L2 captions: {len(L2_captions)}")

# Run reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task_detail, global_sg, llm_prompter, prompt_template)
print(f"Completed reasoning. Predicted failure reason: {reasoning_dict['pred_failure_reason']}, Predicted failure step: {reasoning_dict['pred_failure_step']}")

# Run replanning
replan_dict = run_replanning(task_detail, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print(f"Completed replanning. Suggested correction plan (raw): {replan_dict['llm_plan_raw']}")

# Run correction
correction_dict = run_episode_correction(episode.name, episode.task_type, data_path, task_detail, events, reasoning_dict, replan_dict)
print(f"Completed correction. Correction success: {correction_dict.get('correction_success', False)}")

print_episode_analysis(episode.name, task_detail, reasoning_dict, replan_dict)

In [ ]:
episode = episodes[14]
episode.loc = True
episode.exp = True
episode.co_plan = False
episode.pipeline_error = "The LLM missed putting down the bowl"

In [ ]:
print(len(episodes))
loc_audio_after_fix = sum(ep.loc for ep in episodes)
exp_audio_after_fix = sum(ep.exp for ep in episodes)
co_plan_after_fix = sum(ep.co_plan for ep in episodes)

In [ ]:
episodes